# ☀️ Solar GHI Global Model — Training & 5-Year Solar Decision Support

**Pipeline Overview**

| Phase | What Happens |
|---|---|
| **Part 1** | Load all NSRDB data (4 locations × 5 years), train a single global Random Forest model |
| **Part 2** | Feed an unseen dataset → predict next 5 years of GHI (hourly) |
| **Part 3** | User inputs → full 8-step solar analysis (feasibility or performance optimisation) |

---
**Dataset:** NREL NSRDB  |  Denver CO · Los Angeles CA · Phoenix AZ · Miami FL  |  2020–2024  
**Target:** GHI — Global Horizontal Irradiance (W/m²)  
**Model:** Random Forest Regressor (global, trained across all locations)

---
## PART 1 — Global Model Training (2020–2024)
---

### Section 1 · Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
from pathlib import Path
import pickle

# ── Data / ML ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
%matplotlib inline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Paths ─────────────────────────────────────────────────────────────────────
# Mount Google Drive to access datasets
from google.colab import drive
drive.mount('/content/drive')

# Adjust DATASET_BASE to wherever your dataset folder lives
# Assuming 'dataset' folder is directly in your Google Drive 'My Drive'
DATASET_BASE = #set this to your dataset path
OUTPUT_DIR   = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Feature / target definition ───────────────────────────────────────────────
FEATURE_COLUMNS = [
    'Year', 'Month', 'Day', 'Hour', 'Minute',
    'Temperature', 'Clearsky GHI', 'Relative Humidity',
    'Solar Zenith Angle', 'Wind Speed',
]
TARGET_COLUMN = 'GHI'

# ── Sampling (0.15 = ~26k rows for fast runs; set 1.0 for full dataset) ───────
SAMPLE_FRACTION = 0.15

print('✅ Configuration loaded.')
print(f'   Dataset base : {DATASET_BASE.resolve()}')
print(f'   Output dir   : {OUTPUT_DIR.resolve()}')
print(f'   Sample frac  : {SAMPLE_FRACTION}')

### Section 2 · Dataset Registry

We define the 4 NSRDB locations with their actual coordinates extracted from file metadata.  
Each entry stores all yearly CSV paths for 2020–2024.

In [ ]:
LOCATION_META = {
    'Denver_Colorado': {
        'lat': 39.74, 'lon': -104.99, 'tz': 'Etc/GMT+7', 'alt': 1608,
        'files': [DATASET_BASE / 'Denver_Colorado' / f'967156_39.74_-104.99_{y}.csv'
                  for y in range(2020, 2025)],
    },
    'Los_Angeles_California': {
        'lat': 34.08, 'lon': -118.23, 'tz': 'Etc/GMT+8', 'alt': 100,
        'files': [DATASET_BASE / 'Los_Angeles_California' / f'211947_34.08_-118.23_{y}.csv'
                  for y in range(2020, 2025)],
    },
    'Phoenix_Arizona': {
        'lat': 33.54, 'lon': -112.13, 'tz': 'Etc/GMT+7', 'alt': 361,
        'files': [DATASET_BASE / 'Phoenix_Arizona' / f'514485_33.54_-112.13_{y}.csv'
                  for y in range(2020, 2025)],
    },
    'Miami_Florida': {
        'lat': 25.80, 'lon': -80.21, 'tz': 'Etc/GMT+5', 'alt': 2,
        'files': [DATASET_BASE / 'Miami_Florida' / f'2531375_25.80_-80.21_{y}.csv'
                  for y in range(2020, 2025)],
    },
}

print(f'✅ {len(LOCATION_META)} locations registered:')
for name, meta in LOCATION_META.items():
    print(f'   {name:30s}  lat={meta["lat"]}  lon={meta["lon"]}  {len(meta["files"])} files')

### Section 3 · Data Loading

NSRDB CSVs have **2 header rows** (row 0 = source metadata, row 1 = units).  
We skip both and read from row 2 onward, then tag each row with its location name.

In [ ]:
def load_nsrdb_csv(filepath: Path, skip_rows: int = 2) -> pd.DataFrame:
    """Load a single NSRDB CSV, skipping the 2-row metadata header."""
    return pd.read_csv(filepath, skiprows=skip_rows)


def load_all_locations(location_meta: dict) -> pd.DataFrame:
    """Load and concatenate all CSV files across all registered locations."""
    frames = []
    for loc_name, meta in location_meta.items():
        loc_frames = []
        for filepath in meta['files']:
            if not filepath.exists():
                print(f'  [WARN] Not found: {filepath}')
                continue
            df = load_nsrdb_csv(filepath)
            df['Location'] = loc_name
            loc_frames.append(df)
        if loc_frames:
            loc_df = pd.concat(loc_frames, ignore_index=True)
            frames.append(loc_df)
            print(f'  ✔ {loc_name:30s}  →  {len(loc_df):>7,} rows')
        else:
            print(f'  ✘ {loc_name} — no files loaded')

    if not frames:
        raise FileNotFoundError('No data loaded. Check DATASET_BASE path.')
    return pd.concat(frames, ignore_index=True)


print('Loading historical data (2020–2024)...')
raw_df = load_all_locations(LOCATION_META)
print(f'\n📦 Total rows loaded : {len(raw_df):,}')
print(f'   Columns          : {list(raw_df.columns)}')
raw_df.head(3)

### Section 4 · Preprocessing

Steps applied:
1. Select only the required feature + target columns  
2. Drop rows with any null values  
3. Remove physically impossible rows where `GHI < 0`  
4. Optional sub-sampling for faster iteration

In [ ]:
def preprocess(df: pd.DataFrame, feature_cols: list, target_col: str) -> pd.DataFrame:
    """Select columns, drop nulls, enforce GHI >= 0."""
    required = feature_cols + [target_col]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Missing columns in dataset: {missing_cols}')

    df = df[required + ['Location']].copy()

    before = len(df)
    df = df.dropna()
    after_null = len(df)

    df = df[df[target_col] >= 0].copy()
    after_neg = len(df)

    print(f'  Rows before   : {before:>8,}')
    print(f'  After dropna  : {after_null:>8,}  (removed {before - after_null:,} null rows)')
    print(f'  After GHI>=0  : {after_neg:>8,}  (removed {after_null - after_neg:,} negative-GHI rows)')
    return df.reset_index(drop=True)


print('Preprocessing...')
clean_df = preprocess(raw_df, FEATURE_COLUMNS, TARGET_COLUMN)

# ── Optional sub-sampling for faster runs ─────────────────────────────────────
if SAMPLE_FRACTION < 1.0:
    model_df = clean_df.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'\n  Sub-sampled   : {len(model_df):>8,} rows ({SAMPLE_FRACTION*100:.0f}% of {len(clean_df):,})')
else:
    model_df = clean_df.copy()

print(f'\n✅ Final training dataset shape: {model_df.shape}')
model_df[FEATURE_COLUMNS + [TARGET_COLUMN]].describe()

### Section 5 · Train / Test Split

In [ ]:
X = model_df[FEATURE_COLUMNS].copy()
y = model_df[TARGET_COLUMN].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f'✅ Split complete')
print(f'   Train : {len(X_train):>7,} rows  ({len(X_train)/len(X)*100:.0f}%)')
print(f'   Test  : {len(X_test):>7,} rows  ({len(X_test)/len(X)*100:.0f}%)')

### Section 6 · Train Global Random Forest Model

**Why Random Forest?**  
In our comparative study across 11 models on this dataset, Random Forest achieved the best performance:
- **R² = 0.9128** (explains 91.3% of GHI variance)
- **RMSE = 89.33 W/m²** (lowest error)
- **MAE = 39.69 W/m²**

It also provides built-in **uncertainty quantification** (individual tree predictions) required for best/worst-case scenario forecasting.

**Why "Global" model?**  
Training on all 4 locations together gives the model exposure to a diverse range of climates (arid, coastal, humid, high-altitude). This makes it generalisable to new, unseen locations.

In [ ]:
global_model = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(
        n_estimators    = 200,        # 200 trees → stable predictions
        max_depth       = None,       # fully grown trees capture non-linearity
        min_samples_leaf= 5,          # prevents overfitting on noisy hours
        n_jobs          = -1,         # use all available CPU cores
        random_state    = RANDOM_STATE,
    ))
])

print('Training global Random Forest model...')
global_model.fit(X_train, y_train)
print('✅ Training complete.')

### Section 7 · Model Evaluation on Test Set

In [ ]:
def compute_metrics(y_true, y_pred) -> dict:
    return {
        'MAE' : mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2'  : r2_score(y_true, y_pred),
    }


y_pred_test = global_model.predict(X_test)
metrics = compute_metrics(y_test, y_pred_test)

print('=' * 45)
print('  GLOBAL MODEL — TEST SET PERFORMANCE')
print('=' * 45)
print(f'  MAE  : {metrics["MAE"]:>8.4f}  W/m²')
print(f'  RMSE : {metrics["RMSE"]:>8.4f}  W/m²')
print(f'  R²   : {metrics["R2"]:>8.4f}')
print('=' * 45)

# ── Predicted vs Actual scatter ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test.values, y_pred_test, alpha=0.25, s=6, color='#2980b9', edgecolors='none')
max_val = max(y_test.max(), y_pred_test.max())
ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual GHI (W/m²)', fontsize=12)
ax.set_ylabel('Predicted GHI (W/m²)', fontsize=12)
ax.set_title(
    f'Global Random Forest — Predicted vs Actual\nR²={metrics["R2"]:.4f}   RMSE={metrics["RMSE"]:.2f} W/m²',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'global_model_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

### Section 8 · Feature Importance

In [ ]:
rf_step = global_model.named_steps['rf']
importances = rf_step.feature_importances_
indices = np.argsort(importances)[::-1]

importance_df = pd.DataFrame({
    'Feature'   : [FEATURE_COLUMNS[i] for i in indices],
    'Importance': importances[indices],
})

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#27ae60' if i == 0 else '#3498db' for i in range(len(FEATURE_COLUMNS))]
bars = ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1], color=colors[::-1])
for bar, val in zip(bars, importance_df['Importance'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importance — Global Random Forest', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop features:')
print(importance_df.to_string(index=False))

### Section 9 · Save Global Model to Disk

The trained model is serialised with `pickle` so Part 2 can reload it without retraining.

In [ ]:
MODEL_PATH = OUTPUT_DIR / 'global_rf_model.pkl'

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(global_model, f)

print(f'✅ Global model saved → {MODEL_PATH}')
print(f'   File size: {MODEL_PATH.stat().st_size / 1024:.1f} KB')

---
## PART 2 — Load Unseen Dataset & Predict Next 5 Years (2025–2029)

> **Instructions:** Place your new unseen CSV file(s) in the `unseen_dataset/` folder  
> and set `UNSEEN_LAT`, `UNSEEN_LON`, `UNSEEN_TZ` to match the new location's coordinates.  
> The global model predicts GHI; Solar Zenith Angle for 2025–2029 is computed  
> deterministically using the **NREL Solar Position Algorithm** (implemented below  
> without pvlib, using the standard astronomical formula).

---

### Section 10 · Load Unseen Dataset

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ► USER — FILL IN PATH and optionally verify UNSEEN_TZ_OFFSET
# ═══════════════════════════════════════════════════════════════════════════════

UNSEEN_DATA_DIR  = #put your own unseed data path here

# The following geo-coordinates and location name will be extracted from the CSV
# metadata if available. These are initial defaults/placeholders.
UNSEEN_LAT       = 31.48    # latitude  (degrees N)
UNSEEN_LON       = -99.33  # longitude (degrees E, negative = West)
UNSEEN_ALT       = 443       # altitude in metres
UNSEEN_LOC_NAME  = 'BHARAT' # Location name (will be overwritten by Location ID from file)

# UTC offset is often difficult to infer robustly from timezone strings.
# Please VERIFY this value is correct for your location and season.
# (e.g., -7 for MST, -8 for PST, -6 for CST, -5 for EST during Standard Time)
UNSEEN_TZ_OFFSET = -8       # UTC offset (e.g. -7 for MST, -8 for PST) (will be overwritten from file)

FORECAST_YEARS   = [2025, 2026, 2027, 2028, 2029]

# ═══════════════════════════════════════════════════════════════════════════════

# Load unseen data
unseen_raw_frames = []
if UNSEEN_DATA_DIR.is_dir():
    # Find all CSV files in the directory
    # User specified 5 files from 2020 to 2024.
    # It is good practice to sort them to ensure consistent loading order.
    csv_files = sorted(list(UNSEEN_DATA_DIR.glob('*.csv')))

    if csv_files:
        print(f'Found {len(csv_files)} CSV files in {UNSEEN_DATA_DIR.name}/:')
        # Extract metadata from the first file found (assuming consistency across files in the folder)
        try:
            first_file = csv_files[0]
            # Read the first two lines to get metadata header and values
            metadata_full = pd.read_csv(first_file, nrows=2, header=None, engine='python')
            metadata_values = metadata_full.iloc[1].astype(str).str.strip('"').tolist()

            # Assuming new NSRDB header format based on user's example:
            # Line 1: Source, Location ID, City, State, Country, Latitude, Longitude, Time Zone, Elevation, Local Time Zone
            # Line 2: NSRDB, 1411912, -, -, -, 31.48, -99.33, -6, 443, -6
            UNSEEN_LOC_NAME  = metadata_values[1] # Location ID, e.g., "1411912"
            UNSEEN_LAT       = float(metadata_values[5])
            UNSEEN_LON       = float(metadata_values[6])
            UNSEEN_TZ_OFFSET = int(metadata_values[7]) # Assuming direct integer offset like -6
            UNSEEN_ALT       = int(float(metadata_values[8])) # Elevation can be float sometimes

            print(f'✅ Extracted location metadata from {first_file.name}:')
            print(f'   Location Name (ID) : {UNSEEN_LOC_NAME}')
            print(f'   Latitude           : {UNSEEN_LAT}°')
            print(f'   Longitude          : {UNSEEN_LON}°')
            print(f'   Altitude           : {UNSEEN_ALT} m')
            print(f'   UTC Offset         : {UNSEEN_TZ_OFFSET} hours')

            # Load data from all identified CSV files
            for fpath in csv_files:
                # The actual data starts after the first two metadata rows, using the 3rd row as header
                df = pd.read_csv(fpath, skiprows=2)
                df['Location'] = UNSEEN_LOC_NAME
                unseen_raw_frames.append(df)

            if unseen_raw_frames:
                unseen_raw = pd.concat(unseen_raw_frames, ignore_index=True)
                print(f'\n✅ Unseen dataset loaded from {len(csv_files)} files: {len(unseen_raw):,} rows')
                print(f'   Columns: {list(unseen_raw.columns)}')
                print(f'   Years  : {sorted(unseen_raw["Year"].unique())}')
                display(unseen_raw.head(3))
            else:
                raise FileNotFoundError(f"No data frames created from files in {UNSEEN_DATA_DIR}")

        except Exception as e:
            print(f"[ERROR] Failed to extract metadata or load files from {UNSEEN_DATA_DIR}: {e}")
            print("         Falling back to demo data.")
            # Fallback to demo data if parsing fails
            unseen_raw = pd.read_csv(
                DATASET_BASE / 'Denver_Colorado' / '967156_39.74_-104.99_2024.csv',
                skiprows=2
            )
            unseen_raw['Location'] = 'Demo_Unseen'
            UNSEEN_LAT, UNSEEN_LON, UNSEEN_TZ_OFFSET = 39.74, -104.99, -7
            print(f'   Loaded {len(unseen_raw):,} rows as demo unseen dataset.')
    else:
        print(f'[INFO] No CSV files found in {UNSEEN_DATA_DIR} — using demo data.')
        unseen_raw = pd.read_csv(
            DATASET_BASE / 'Denver_Colorado' / '967156_39.74_-104.99_2024.csv',
            skiprows=2
        )
        unseen_raw['Location'] = 'Demo_Unseen'
        UNSEEN_LAT, UNSEEN_LON, UNSEEN_TZ_OFFSET = 39.74, -104.99, -7
        print(f'   Loaded {len(unseen_raw):,} rows as demo unseen dataset.')

else:
    # ── Demo fallback: reuse Denver 2024 as a stand-in for unseen data ──────
    print('[INFO] Unseen data directory not found or is not a directory — using demo data.')
    unseen_raw = pd.read_csv(
        DATASET_BASE / 'Denver_Colorado' / '967156_39.74_-104.99_2024.csv',
        skiprows=2
    )
    unseen_raw['Location'] = 'Demo_Unseen'
    UNSEEN_LAT, UNSEEN_LON, UNSEEN_TZ_OFFSET = 39.74, -104.99, -7
    print(f'   Loaded {len(unseen_raw):,} rows as demo unseen dataset.')


### Section 11 · Solar Zenith Angle for Future Years

Solar Zenith Angle (SZA) is a **purely astronomical quantity** — it depends only on time and geographic coordinates, not on weather. For 2025–2029 we compute it analytically using the standard NREL Solar Position Algorithm formula.

The formula computes the sun's declination and hour angle from the day of year and local solar time, then calculates the zenith angle from the observer's latitude.

In [ ]:
def compute_solar_zenith_angle(
    year: int, month: int, day: int, hour: int, minute: int,
    lat_deg: float, lon_deg: float, utc_offset: int
) -> float:
    """
    Compute Solar Zenith Angle (degrees) using the standard astronomical formula.

    Based on: Iqbal, M. (1983) An Introduction to Solar Radiation.
    Accurate to ~0.5° for typical solar energy applications.

    Parameters
    ----------
    year, month, day, hour, minute : date/time components (local time)
    lat_deg    : latitude  in decimal degrees (N positive)
    lon_deg    : longitude in decimal degrees (E positive, W negative)
    utc_offset : local UTC offset (e.g. -7 for MST)

    Returns
    -------
    zenith_angle : float — Solar Zenith Angle in degrees (0°=overhead, 90°=horizon)
    """
    import math

    # Day of year
    doy = pd.Timestamp(year, month, day).day_of_year

    # Fractional hour in local time
    local_hour = hour + minute / 60.0

    # Convert local time → solar time (equation of time correction + longitude correction)
    B     = 2 * math.pi * (doy - 1) / 365.0
    eot   = 229.18 * (0.000075 + 0.001868 * math.cos(B)
                      - 0.032077 * math.sin(B)
                      - 0.014615 * math.cos(2*B)
                      - 0.04089  * math.sin(2*B))  # minutes
    lstm  = 15 * utc_offset                         # local standard time meridian
    tc    = 4 * (lon_deg - lstm) + eot              # time correction (minutes)
    solar_hour = local_hour + tc / 60.0

    # Hour angle (15° per hour from solar noon)
    hour_angle = math.radians(15.0 * (solar_hour - 12.0))

    # Solar declination
    declination = math.radians(23.45 * math.sin(math.radians(360 * (284 + doy) / 365.0)))

    # Zenith angle
    lat_rad   = math.radians(lat_deg)
    cos_theta = (math.sin(lat_rad) * math.sin(declination)
                 + math.cos(lat_rad) * math.cos(declination) * math.cos(hour_angle))
    cos_theta = max(-1.0, min(1.0, cos_theta))     # clamp numerical errors
    return math.degrees(math.acos(cos_theta))


# Vectorised wrapper
def add_solar_zenith_to_df(df: pd.DataFrame, lat: float, lon: float, tz: int) -> pd.DataFrame:
    """Compute and attach Solar Zenith Angle for every row in df."""
    df = df.copy()
    df['Solar Zenith Angle'] = [
        compute_solar_zenith_angle(
            int(r.Year), int(r.Month), int(r.Day),
            int(r.Hour), int(r.Minute), lat, lon, tz
        )
        for r in df[['Year','Month','Day','Hour','Minute']].itertuples(index=False)
    ]
    return df


print('✅ Solar Zenith Angle function ready.')
# Quick sanity check — noon on June 21 in Denver (SZA should be ~26°)
test_sza = compute_solar_zenith_angle(2025, 6, 21, 12, 0, 39.74, -104.99, -7)
print(f'   Sanity check (Denver, June 21, 12:00): SZA = {test_sza:.2f}° (expect ~26°)')

### Section 12 · Build Climatology & Generate Future Feature Grid

Weather features (Temperature, Humidity, Wind Speed, Clearsky GHI) for 2025–2029 are filled using **climatological means** — the average value of each feature for a given (Month, Hour) pair across all available historical years. This is standard practice when future NWP data is not available.

In [ ]:
WEATHER_COLS = ['Temperature', 'Clearsky GHI', 'Relative Humidity', 'Wind Speed']


def build_climatology(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute (Month, Hour) mean of weather features from historical data.
    Used to fill future forecast rows where real data is unavailable.
    """
    available = [c for c in WEATHER_COLS if c in df.columns]
    clim = df.groupby(['Month', 'Hour'])[available].mean().reset_index()
    return clim


def build_future_features(
    forecast_years: list,
    climatology: pd.DataFrame,
    lat: float,
    lon: float,
    tz_offset: int,
) -> pd.DataFrame:
    """
    Build hourly feature matrix for each forecast year.
    - Time grid   : every hour (30-min offset matching NSRDB convention)
    - SZA         : computed analytically
    - Weather     : filled from climatology
    """
    all_frames = []
    for year in forecast_years:
        times = pd.date_range(
            start=f'{year}-01-01 00:30',
            end  =f'{year}-12-31 23:30',
            freq ='1h'
        )
        df_yr = pd.DataFrame({
            'Year'  : times.year,
            'Month' : times.month,
            'Day'   : times.day,
            'Hour'  : times.hour,
            'Minute': times.minute,
        })
        # Fill weather from climatology
        df_yr = df_yr.merge(climatology, on=['Month', 'Hour'], how='left')
        all_frames.append(df_yr)

    future_df = pd.concat(all_frames, ignore_index=True)

    # Compute Solar Zenith Angle for all rows
    print(f'  Computing Solar Zenith Angle for {len(future_df):,} future timesteps...')
    future_df = add_solar_zenith_to_df(future_df, lat, lon, tz_offset)
    return future_df


print('Building climatology from unseen dataset...')
unseen_clean = preprocess(unseen_raw, FEATURE_COLUMNS, TARGET_COLUMN)
climatology  = build_climatology(unseen_clean)
print(f'✅ Climatology shape: {climatology.shape}  (12 months × 24 hours = 288 rows)')

print('\nGenerating future feature grid (2025–2029)...')
future_df = build_future_features(
    FORECAST_YEARS, climatology, UNSEEN_LAT, UNSEEN_LON, UNSEEN_TZ_OFFSET
)
print(f'✅ Future grid: {len(future_df):,} rows  ({len(FORECAST_YEARS)} years × ~8,760 hours)')
future_df.head(3)

### Section 13 · Predict GHI with Best / Worst / Average Scenarios

Random Forest exposes individual tree predictions via `estimators_`. Collecting all tree outputs lets us compute:
- **Mean** → point prediction (best estimate)
- **10th percentile** → worst-case scenario (unfavourable conditions)
- **90th percentile** → best-case scenario (favourable conditions)

In [ ]:
def predict_with_scenarios(pipeline: Pipeline, X: pd.DataFrame):
    """
    Predict GHI with uncertainty bands from individual Random Forest trees.

    Returns
    -------
    mean_pred  : average across all trees  (main forecast)
    lower_pred : 10th percentile           (worst case)
    upper_pred : 90th percentile           (best case)
    """
    scaler   = pipeline.named_steps['scaler']
    rf       = pipeline.named_steps['rf']
    X_scaled = scaler.transform(X)

    # Each tree gives one prediction per sample → shape (n_trees, n_samples)
    tree_preds = np.array([tree.predict(X_scaled) for tree in rf.estimators_])

    mean_pred  = np.clip(tree_preds.mean(axis=0),                    0, None)
    lower_pred = np.clip(np.percentile(tree_preds, 10, axis=0),      0, None)
    upper_pred = np.clip(np.percentile(tree_preds, 90, axis=0),      0, None)
    return mean_pred, lower_pred, upper_pred


# Reload model (safe even if notebook is restarted from Part 2)
with open(MODEL_PATH, 'rb') as f:
    global_model = pickle.load(f)

print('Running predictions for 2025–2029...')
X_future = future_df[FEATURE_COLUMNS]
mean_pred, lower_pred, upper_pred = predict_with_scenarios(global_model, X_future)

future_df['GHI_mean']  = mean_pred
future_df['GHI_lower'] = lower_pred
future_df['GHI_upper'] = upper_pred

print('\n  Yearly GHI Summary (W/m²):')
print(f'  {"Year":>4}  {"Avg":>8}  {"Worst":>8}  {"Best":>8}  {"Ann. Energy (Wh/m²)":>20}')
print('  ' + '-'*58)
yearly_summary = {}
for yr in FORECAST_YEARS:
    mask = future_df['Year'] == yr
    avg  = future_df.loc[mask, 'GHI_mean'].mean()
    lo   = future_df.loc[mask, 'GHI_lower'].mean()
    hi   = future_df.loc[mask, 'GHI_upper'].mean()
    ann  = future_df.loc[mask, 'GHI_mean'].sum()
    yearly_summary[yr] = {'avg': avg, 'lo': lo, 'hi': hi, 'annual_wh': ann}
    print(f'  {yr:>4}  {avg:>8.1f}  {lo:>8.1f}  {hi:>8.1f}  {ann:>20,.0f}')
print()
print('✅ Prediction complete.')

### Section 14 · 5-Year Forecast Visualisation

In [ ]:
# ── Monthly average GHI across forecast years ─────────────────────────────────
monthly_avg = future_df.groupby(['Year', 'Month'])[['GHI_mean', 'GHI_lower', 'GHI_upper']].mean().reset_index()
monthly_avg['date'] = pd.to_datetime(monthly_avg[['Year', 'Month']].assign(Day=15))

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ── Top: monthly line + shaded band ───────────────────────────────────────────
ax = axes[0]
ax.fill_between(
    monthly_avg['date'],
    monthly_avg['GHI_lower'],
    monthly_avg['GHI_upper'],
    alpha=0.25, color='#e67e22', label='Worst–Best range'
)
ax.plot(monthly_avg['date'], monthly_avg['GHI_mean'],
        color='#e67e22', lw=2, label='Average prediction')
# Year boundary lines
for yr in FORECAST_YEARS[1:]:
    ax.axvline(pd.Timestamp(f'{yr}-01-01'), color='grey', lw=0.7, ls='--', alpha=0.6)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Monthly Avg GHI (W/m²)', fontsize=11)
ax.set_title(f'5-Year GHI Forecast (2025–2029) — {UNSEEN_LOC_NAME}\nShaded band = Worst/Best case range',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ── Bottom: annual energy bar chart ───────────────────────────────────────────
ax2 = axes[1]
yrs  = list(yearly_summary.keys())
avgs = [yearly_summary[y]['annual_wh'] / 1000 for y in yrs]  # Wh → kWh/m²
los  = [yearly_summary[y]['lo'] * 8760 / 1000 for y in yrs]
his  = [yearly_summary[y]['hi'] * 8760 / 1000 for y in yrs]

x = np.arange(len(yrs))
bars = ax2.bar(x, avgs, color='#2980b9', alpha=0.8, zorder=3, label='Average')
ax2.errorbar(
    x, avgs,
    yerr=[np.array(avgs) - np.array(los), np.array(his) - np.array(avgs)],
    fmt='none', color='black', capsize=8, lw=2, zorder=4, label='Worst–Best range'
)
ax2.set_xticks(x)
ax2.set_xticklabels([str(y) for y in yrs], fontsize=12)
ax2.set_ylabel('Annual Solar Energy (kWh/m²)', fontsize=11)
ax2.set_title('Estimated Annual Solar Energy per m² of Panel — with Uncertainty Range',
              fontsize=12, fontweight='bold')
for bar, val in zip(bars, avgs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:.0f}', ha='center', fontsize=10, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'forecast_5year.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

---
## PART 3 — User Input & Full Solar Analysis Pipeline

This part implements the complete 8-step user-driven workflow:

| Step | Description |
|---|---|
| **Step 1** | Collect user inputs (install status, area, consumption, tilt) |
| **Step 2** | Retrieve climate & irradiance data from NREL NSRDB |
| **Step 3** | Predict solar power generation (hour / day / year) |
| **Step 4** | Calculate expected energy output using PV formula |
| **Step 5** | Panel tilt sensitivity analysis (0°–60°) |
| **Step 6** | Scenario-based forecasting (best / worst / average) |
| **Step 7** | Prediction reliability analysis (hour-ahead vs day-ahead) |
| **Step 8** | Final output report + dashboard |

Two analysis paths based on user response:
- **`plan`** → Feasibility analysis — should I install solar panels?
- **`installed`** → Performance & optimisation — am I getting the most from my panels?
- **`no`** → Skip analysis

---

### Section 15 · Step 1 — User Input

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ► USER — FILL IN YOUR DETAILS HERE
# ═══════════════════════════════════════════════════════════════════════════════

# Question: Do you plan to install solar panels, or have you already installed them?
# Enter 'plan' if you are planning to install.
# Enter 'installed' if you have already installed and want analysis based on existing setup.
# Enter 'no' to skip feasibility analysis.

analysis_type_response = input("Do you plan to install solar panels, or have you already installed them? (plan / installed / no): ").strip().lower()

PLANNING_TO_INSTALL = (analysis_type_response == 'plan')
ALREADY_INSTALLED   = (analysis_type_response == 'installed')

USER_AREA_M2                = None
USER_YEARLY_CONSUMPTION_KWH = None
# New variables for already installed scenario
PANEL_TILT_ANGLE_DEG        = None
INSTALLED_SYSTEM_CAPACITY_KW = None
USER_DAILY_CONSUMPTION_KWH  = None # Will be converted to USER_YEARLY_CONSUMPTION_KWH

if PLANNING_TO_INSTALL:
    print("\n--- Planning to install solar panels ---")
    USER_AREA_M2 = float(input("Enter the area available for solar panels (in m²): ").strip())
    USER_YEARLY_CONSUMPTION_KWH = float(input("Enter your average yearly household power consumption (in kWh): ").strip())

elif ALREADY_INSTALLED:
    print("\n--- Analyzing existing solar panel installation ---")
    PANEL_TILT_ANGLE_DEG = float(input("Enter your panel tilt angle (in degrees): ").strip())
    USER_AREA_M2 = float(input("Enter the total area of your solar panels (in m²): ").strip())
    INSTALLED_SYSTEM_CAPACITY_KW = float(input("Enter your installed system capacity (in kW): ").strip())
    USER_DAILY_CONSUMPTION_KWH = float(input("Enter your average daily household consumption (in kWh): ").strip())
    USER_YEARLY_CONSUMPTION_KWH = USER_DAILY_CONSUMPTION_KWH * 365 # Convert for consistency

# ── Fixed solar panel parameters (standard commercial panels) ─────────────────
# These values are used in the feasibility calculation. If ALREADY_INSTALLED,
# the 'installed capacity' and 'tilt angle' are recorded but current model
# still uses these fixed efficiency/performance for prediction.
PANEL_EFFICIENCY      = 0.20   # 20% — typical monocrystalline silicon panel
PERFORMANCE_RATIO     = 0.80   # system losses: inverter, wiring, soiling (~80%)

# ── Which forecast year to base feasibility on ────────────────────────────────
USE_AVERAGE_ALL_YEARS = True

# ═══════════════════════════════════════════════════════════════════════════════

# Control flag for subsequent sections (s17, s18) which currently check PLANNING_TO_INSTALL
# We'll re-purpose PLANNING_TO_INSTALL to mean "perform some kind of solar analysis"
_perform_any_solar_analysis = PLANNING_TO_INSTALL or ALREADY_INSTALLED

if _perform_any_solar_analysis:
    print('\n✅ User requested solar analysis. Proceeding to feasibility analysis.')
    if PLANNING_TO_INSTALL:
        print(f'   Analysis type           : Planning to install')
        print(f'   Area available          : {USER_AREA_M2} m²')
        print(f'   Yearly consumption      : {USER_YEARLY_CONSUMPTION_KWH:,.0f} kWh/year')
    elif ALREADY_INSTALLED:
        print(f'   Analysis type           : Already installed')
        print(f'   Panel tilt angle        : {PANEL_TILT_ANGLE_DEG}°')
        print(f'   Panel area              : {USER_AREA_M2} m²')
        print(f'   System capacity         : {INSTALLED_SYSTEM_CAPACITY_KW} kW')
        print(f'   Daily consumption       : {USER_DAILY_CONSUMPTION_KWH:,.0f} kWh/day')
        print(f'   Estimated yearly consumption: {USER_YEARLY_CONSUMPTION_KWH:,.0f} kWh/year')
    print(f'   Panel efficiency used   : {PANEL_EFFICIENCY*100:.0f}%')
    print(f'   System performance ratio: {PERFORMANCE_RATIO*100:.0f}%')
else:
    print('ℹ️  No solar analysis requested. Skipping feasibility analysis.')

# Set PLANNING_TO_INSTALL for downstream cells which expect this variable
PLANNING_TO_INSTALL = _perform_any_solar_analysis
_perform_analysis = _perform_any_solar_analysis

### Section 16 · Step 2 — Retrieve Climate & Irradiance Data

Historical irradiance and meteorological data are retrieved from the **NREL NSRDB dataset (2020–2024)**.  
The `future_df` table (built in Part 2) holds hourly GHI predictions with Solar Zenith Angle  
computed via the NREL Solar Position Algorithm.  
This section also extracts the next-hour and next-day slices needed for Step 3.

In [ ]:
if not PLANNING_TO_INSTALL:
    print('ℹ️  Skipping — no analysis requested.')
else:
    import datetime

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 2 — RETRIEVE CLIMATE & IRRADIANCE DATA
    #
    # future_df was built in Part 2 and contains:
    #   • Hourly rows for 2025–2029 (43,800 rows total)
    #   • Weather features filled from 2020–2024 climatological means
    #   • Solar Zenith Angle computed via NREL SPA formula
    #   • GHI_mean / GHI_lower / GHI_upper from the Random Forest ensemble
    #
    # This section:
    #   1. Prints a full data summary for transparency
    #   2. Shows per-year GHI statistics
    #   3. Shows monthly seasonal pattern
    #   4. Extracts next-hour and next-day slices for Step 3,
    #      anchored to the current real-world datetime
    # ══════════════════════════════════════════════════════════════════════════

    # ── 2a. Data source summary ───────────────────────────────────────────────
    print('STEP 2 — Climate & Irradiance Data Retrieved')
    print('=' * 60)
    print(f'  Source           : NREL NSRDB (2020–2024, 4 locations)')
    print(f'  Forecast site    : {UNSEEN_LOC_NAME}')
    print(f'  Coordinates      : Lat {UNSEEN_LAT}°  Lon {UNSEEN_LON}°')
    print(f'  Forecast window  : {FORECAST_YEARS[0]} – {FORECAST_YEARS[-1]}  (5 years)')
    print(f'  Total timesteps  : {len(future_df):,}  (hourly)')
    print(f'  Features present : {", ".join(FEATURE_COLUMNS[:5])}')
    print(f'                     {", ".join(FEATURE_COLUMNS[5:])}')
    print(f'  GHI columns      : GHI_mean  (average prediction — mean of 200 trees)')
    print(f'                     GHI_lower (worst case  — 10th percentile of trees)')
    print(f'                     GHI_upper (best case   — 90th percentile of trees)')
    print(f'  SZA source       : NREL Solar Position Algorithm (pure astronomy)')
    print('=' * 60)

    # ── 2b. Per-year GHI statistics ───────────────────────────────────────────
    print()
    print('  Per-Year GHI Statistics (W/m²):')
    print(f'  {"Year":>4}  {"Mean GHI":>9}  {"Peak GHI":>9}  {"Daytime Hrs":>12}  {"Annual Wh/m²":>14}')
    print('  ' + '-' * 57)
    for yr in FORECAST_YEARS:
        mask        = future_df['Year'] == yr
        yr_ghi      = future_df.loc[mask, 'GHI_mean']
        daytime_hrs = int((yr_ghi > 0).sum())
        annual_wh   = yr_ghi.sum()
        print(f'  {yr:>4}  {yr_ghi.mean():>9.2f}  '
              f'{yr_ghi.max():>9.1f}  '
              f'{daytime_hrs:>12,}  '
              f'{annual_wh:>14,.0f}')
    print()

    # ── 2c. Monthly seasonal pattern ─────────────────────────────────────────
    month_names  = ['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec']
    monthly_clim = (
        future_df[future_df['GHI_mean'] > 0]
        .groupby('Month')['GHI_mean']
        .mean()
    )
    print('  Monthly Average Daytime GHI (W/m²) — seasonal pattern:')
    print('  ' + '  '.join(f'{monthly_clim.get(m, 0.0):>6.1f}' for m in range(1, 13)))
    print('  ' + '  '.join(f'{n:>6}' for n in month_names))
    max_ghi  = monthly_clim.max()
    bar_row  = '  '
    for m in range(1, 13):
        val       = monthly_clim.get(m, 0.0)
        bar_len   = int((val / max_ghi) * 8)
        bar_row  += f'  {"▓" * bar_len:<8}'
    print(bar_row)
    print()

    # ── 2d. Smart datetime anchoring ─────────────────────────────────────────
    # Anchors next-hour / next-day slices to the current real-world datetime.
    # Falls back to the first daytime hour of the forecast window if today
    # is outside 2025–2029, or if the matched hour has zero GHI (nighttime).

    now      = datetime.datetime.now()
    now_year = now.year

    if FORECAST_YEARS[0] <= now_year <= FORECAST_YEARS[-1]:
        hour_mask = (
            (future_df['Year']  == now_year)  &
            (future_df['Month'] == now.month) &
            (future_df['Day']   == now.day)   &
            (future_df['Hour']  == now.hour)
        )
        if hour_mask.any():
            anchor_idx = future_df.index[hour_mask][0]
        else:
            daytime_mask = (future_df['Year'] == now_year) & (future_df['GHI_mean'] > 0)
            anchor_idx   = future_df.index[daytime_mask][0]
    else:
        daytime_mask = (
            (future_df['Year'] == FORECAST_YEARS[0]) & (future_df['GHI_mean'] > 0)
        )
        anchor_idx = future_df.index[daytime_mask][0]

    # next_hour_row — single row at anchor
    next_hour_row = future_df.loc[[anchor_idx]]

    # next_day_rows — all 24 hours of the same calendar day as the anchor
    anchor_row = future_df.loc[anchor_idx]
    day_mask   = (
        (future_df['Year']  == int(anchor_row['Year']))  &
        (future_df['Month'] == int(anchor_row['Month'])) &
        (future_df['Day']   == int(anchor_row['Day']))
    )
    next_day_rows = future_df[day_mask]

    # ── 2e. Print extracted slices ────────────────────────────────────────────
    ref_dt = (f'{int(anchor_row["Year"]):d}-{int(anchor_row["Month"]):02d}-'
              f'{int(anchor_row["Day"]):02d}  '
              f'Hour {int(anchor_row["Hour"]):02d}:30')

    print(f'  ── Extracted Slices (anchored to: {ref_dt}) ──')
    print()
    print('  Next-Hour Slice (1 row):')
    display_cols = ['Year','Month','Day','Hour','Solar Zenith Angle',
                    'GHI_mean','GHI_lower','GHI_upper']
    print('  ' + next_hour_row[display_cols].to_string(index=False))
    print()
    print(f'  Next-Day Slice ({len(next_day_rows)} hourly rows — full calendar day):')
    print(f'    GHI_mean  total  : {next_day_rows["GHI_mean"].sum():>9.1f}  Wh/m²')
    print(f'    GHI_lower total  : {next_day_rows["GHI_lower"].sum():>9.1f}  Wh/m²  (worst case)')
    print(f'    GHI_upper total  : {next_day_rows["GHI_upper"].sum():>9.1f}  Wh/m²  (best case)')
    print(f'    Daytime hours    : {int((next_day_rows["GHI_mean"] > 0).sum()):>9}  hrs  (GHI > 0)')
    peak_idx  = next_day_rows['GHI_mean'].idxmax()
    peak_hour = int(next_day_rows.loc[peak_idx, 'Hour'])
    peak_ghi  = next_day_rows.loc[peak_idx, 'GHI_mean']
    print(f'    Peak GHI hour    :  Hour {peak_hour:02d}:30  ({peak_ghi:.1f} W/m²)')
    print()
    print('  Hourly GHI profile for the day (GHI_mean, W/m²):')
    print('  ' + '-' * 50)
    for _, row in next_day_rows.iterrows():
        bar_len = min(int(row['GHI_mean'] / 50), 25)
        print(f'  Hour {int(row["Hour"]):02d}  '
              f'{row["GHI_mean"]:>6.1f} W/m²  '
              f'{"█" * bar_len}')
    print()
    print('✅ Climate data ready. next_hour_row and next_day_rows available for Step 3.')

### Section 17 · Step 3 & 4 — Predict Solar Power Generation & Calculate Energy Output

**Step 3:** The trained global Random Forest model predicts GHI for next-hour, next-day, and yearly horizons.

**Step 4:** Predicted irradiance is converted to power using the PV generation formula:

> **Energy (kWh) = GHI (Wh/m²) × Panel Area (m²) × Efficiency × Performance Ratio ÷ 1000**

In [ ]:
if not PLANNING_TO_INSTALL:
    print('ℹ️  Skipping — no analysis requested.')
else:

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 3 & 4 — PREDICT SOLAR POWER GENERATION & CALCULATE ENERGY OUTPUT
    #
    # Step 3: Use the trained Random Forest model to predict GHI for:
    #         • Next-hour output
    #         • Next-day output
    #         • Yearly aggregated output (2025–2029)
    #
    # Step 4: Convert predicted GHI → usable electricity using PV formula:
    #         Energy (kWh) = GHI (Wh/m²) × Area (m²) × Efficiency × PR / 1000
    # ══════════════════════════════════════════════════════════════════════════

    area = USER_AREA_M2
    eff  = PANEL_EFFICIENCY
    pr   = PERFORMANCE_RATIO
    cons = USER_YEARLY_CONSUMPTION_KWH

    # ── PV generation formula ─────────────────────────────────────────────────
    def ghi_to_kwh(ghi_wh_per_m2: float) -> float:
        """
        PV Generation Formula:
            Energy (kWh) = GHI (Wh/m²) × Area (m²) × Efficiency × Performance Ratio / 1000

        Parameters
        ----------
        ghi_wh_per_m2 : total irradiance over the time period (Wh/m²)
                        for a single hour  → W/m²  (same numerically as Wh/m²)
                        for a full day     → sum of 24 hourly values
                        for a full year    → sum of 8760 hourly values

        Returns
        -------
        float : energy in kWh
        """
        return (ghi_wh_per_m2 * area * eff * pr) / 1000


    # ── NEXT-HOUR prediction ──────────────────────────────────────────────────
    # Single row — GHI value is already in W/m² = Wh for that 1-hour slot
    nh_ghi_mean  = float(next_hour_row['GHI_mean'].values[0])
    nh_ghi_lower = float(next_hour_row['GHI_lower'].values[0])
    nh_ghi_upper = float(next_hour_row['GHI_upper'].values[0])

    nh_kwh_avg   = ghi_to_kwh(nh_ghi_mean)
    nh_kwh_worst = ghi_to_kwh(nh_ghi_lower)
    nh_kwh_best  = ghi_to_kwh(nh_ghi_upper)

    # ── NEXT-DAY prediction ───────────────────────────────────────────────────
    # Sum all 24 hourly GHI values → total Wh/m² for the day
    nd_ghi_mean  = float(next_day_rows['GHI_mean'].sum())
    nd_ghi_lower = float(next_day_rows['GHI_lower'].sum())
    nd_ghi_upper = float(next_day_rows['GHI_upper'].sum())

    nd_kwh_avg   = ghi_to_kwh(nd_ghi_mean)
    nd_kwh_worst = ghi_to_kwh(nd_ghi_lower)
    nd_kwh_best  = ghi_to_kwh(nd_ghi_upper)

    # ── YEARLY prediction (per year + 5-year average) ─────────────────────────
    yearly_rows = []
    for yr in FORECAST_YEARS:
        mask = future_df['Year'] == yr
        ghi_sum_mean  = float(future_df.loc[mask, 'GHI_mean'].sum())
        ghi_sum_lower = float(future_df.loc[mask, 'GHI_lower'].sum())
        ghi_sum_upper = float(future_df.loc[mask, 'GHI_upper'].sum())

        e_avg   = ghi_to_kwh(ghi_sum_mean)
        e_worst = ghi_to_kwh(ghi_sum_lower)
        e_best  = ghi_to_kwh(ghi_sum_upper)

        yearly_rows.append({
            'Year'         : yr,
            'kwh_avg'      : e_avg,
            'kwh_worst'    : e_worst,
            'kwh_best'     : e_best,
            'coverage_%'   : round(min(e_avg   / cons * 100, 100), 1),
            'surplus_kwh'  : round(e_avg - cons, 1),
        })

    yr_df          = pd.DataFrame(yearly_rows)
    avg_yearly_kwh = yr_df['kwh_avg'].mean()
    avg_worst_kwh  = yr_df['kwh_worst'].mean()
    avg_best_kwh   = yr_df['kwh_best'].mean()
    avg_coverage   = round(min(avg_yearly_kwh / cons * 100, 100), 1)

    # ── Print results ─────────────────────────────────────────────────────────
    print('STEP 3 & 4 — Solar Power Generation Predictions')
    print('=' * 62)
    print(f'  Panel area         : {area} m²')
    print(f'  Efficiency         : {eff*100:.0f}%')
    print(f'  Performance ratio  : {pr*100:.0f}%')
    print(f'  Formula            : GHI × {area} × {eff} × {pr} / 1000')
    print('=' * 62)

    if ALREADY_INSTALLED:
        print()
        print(f'  ── Next-Hour Output ──────────────────────────────────')
        print(f'     GHI (W/m²)     :  avg={nh_ghi_mean:.1f}   '
              f'worst={nh_ghi_lower:.1f}   best={nh_ghi_upper:.1f}')
        print(f'     🌤  Average    : {nh_kwh_avg:.5f} kWh')
        print(f'     🌧  Worst case : {nh_kwh_worst:.5f} kWh')
        print(f'     ☀️   Best case  : {nh_kwh_best:.5f} kWh')

        print()
        print(f'  ── Next-Day Output ───────────────────────────────────')
        print(f'     GHI sum (Wh/m²):  avg={nd_ghi_mean:.1f}   '
              f'worst={nd_ghi_lower:.1f}   best={nd_ghi_upper:.1f}')
        print(f'     🌤  Average    : {nd_kwh_avg:.3f} kWh')
        print(f'     🌧  Worst case : {nd_kwh_worst:.3f} kWh')
        print(f'     ☀️   Best case  : {nd_kwh_best:.3f} kWh')
    else:
        print('  ℹ️  Hour/day predictions shown for installed users only.')
        print('  ── Yearly Output is the relevant metric for planning ──')

    print()
    print(f'  ── Yearly Output (2025–2029) ─────────────────────────')
    print(f'  {"Year":>4}  {"Avg (kWh)":>10}  {"Worst (kWh)":>12}  '
          f'{"Best (kWh)":>11}  {"Coverage":>9}  {"Surplus":>10}')
    print('  ' + '-' * 62)
    for _, row in yr_df.iterrows():
        # Corrected format specifier: convert to int before applying comma formatting
        surplus_label = f'+{int(row["surplus_kwh"]):,d}' if row['surplus_kwh'] >= 0 \
                        else f'{int(row["surplus_kwh"]):,d}'
        print(f'  {int(row["Year"]):>4}  '
              f'{row["kwh_avg"]:>10,.1f}  '
              f'{row["kwh_worst"]:>12,.1f}  '
              f'{row["kwh_best"]:>11,.1f}  '
              f'{row["coverage_%"]:>8.1f}%  '
              f'{surplus_label:>10}')

    print('  ' + '-' * 62)
    print(f'  {"Avg":>4}  {avg_yearly_kwh:>10,.1f}  '
          f'{avg_worst_kwh:>12,.1f}  '
          f'{avg_best_kwh:>11,.1f}  '
          f'{avg_coverage:>8.1f}%')

    print()
    print(f'  Yearly consumption : {cons:,.0f} kWh/year')
    surplus_5yr = avg_yearly_kwh - cons
    if surplus_5yr >= 0:
        print(f'  ✅ Solar SURPLUS    : +{int(surplus_5yr):,d} kWh/year on average')
    else:
        print(f'  ⚠️  Grid DEFICIT    : {int(abs(surplus_5yr)):,d} kWh/year still needed from grid')
    print('=' * 62)

### Section 18 · Step 5 — Panel Tilt Sensitivity Analysis

Evaluates how tilt angles from **0°–60°** affect annual energy output using the  
Liu-Jordan cosine transposition formula (south-facing panels assumed).

- **Already installed** users → compares their current tilt vs optimal tilt  
- **Planning** users → recommends the best tilt angle to install at

In [ ]:
if not PLANNING_TO_INSTALL:
    print('ℹ️  Skipping — no analysis requested.')
else:

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 5 — PANEL TILT SENSITIVITY ANALYSIS
    #
    # Evaluates how different tilt angles (0°–60°) affect annual energy output.
    # Applies the Liu-Jordan cosine transposition formula to correct flat-surface
    # GHI predictions for a tilted, south-facing panel.
    #
    # For 'installed' users → compares their current tilt vs optimal
    # For 'plan' users      → recommends the best tilt to install at
    # ══════════════════════════════════════════════════════════════════════════

    # ── Tilt correction formula ───────────────────────────────────────────────
    def apply_tilt_correction(ghi_array: np.ndarray,
                               sza_array: np.ndarray,
                               tilt_deg: float) -> np.ndarray:
        """
        Adjust flat-surface GHI for a tilted south-facing panel.

        Formula : GHI_tilted = GHI × cos(SZA_rad − tilt_rad)
        Negative cosine values (sun behind panel) are clamped to 0.

        Parameters
        ----------
        ghi_array : hourly GHI predictions (W/m²), shape (n,)
        sza_array : Solar Zenith Angle in degrees,  shape (n,)
        tilt_deg  : panel tilt angle in degrees (0° = flat, 90° = vertical)

        Returns
        -------
        np.ndarray : tilt-corrected GHI values, shape (n,)
        """
        cos_aoi = np.cos(np.deg2rad(sza_array) - np.deg2rad(tilt_deg))
        return ghi_array * np.clip(cos_aoi, 0, None)


    # ── Sweep tilt angles 0°–60° ──────────────────────────────────────────────
    ghi_mean_all = future_df['GHI_mean'].values
    sza_all      = future_df['Solar Zenith Angle'].values
    tilt_angles  = list(range(0, 65, 5))   # [0, 5, 10, 15 … 60]

    sweep_rows = []
    for tilt in tilt_angles:
        tilted_ghi  = apply_tilt_correction(ghi_mean_all, sza_all, tilt)
        energy_kwh  = ghi_to_kwh(tilted_ghi.sum())
        sweep_rows.append({
            'tilt_deg'         : tilt,
            'annual_energy_kwh': energy_kwh,
        })

    sensitivity_df = pd.DataFrame(sweep_rows)

    # % change vs flat (0°) baseline
    baseline = sensitivity_df.loc[sensitivity_df['tilt_deg'] == 0, 'annual_energy_kwh'].values[0]
    sensitivity_df['pct_vs_flat'] = (
        (sensitivity_df['annual_energy_kwh'] - baseline) / baseline * 100
    ).round(2)

    # Identify optimal tilt
    optimal_idx    = sensitivity_df['annual_energy_kwh'].idxmax()
    optimal_tilt   = int(sensitivity_df.loc[optimal_idx, 'tilt_deg'])
    optimal_energy = float(sensitivity_df.loc[optimal_idx, 'annual_energy_kwh'])

    # For 'ALREADY_INSTALLED' users, find the closest analyzed tilt angle
    if ALREADY_INSTALLED:
        nearest_row    = sensitivity_df.iloc[
            (sensitivity_df['tilt_deg'] - PANEL_TILT_ANGLE_DEG).abs().argsort().iloc[0]
        ]
        current_energy_at_closest_tilt = float(nearest_row['annual_energy_kwh'])
        current_tilt_analyzed = int(nearest_row['tilt_deg'])
    else:
        current_energy_at_closest_tilt = 0 # Not applicable for planning users
        current_tilt_analyzed = 0

    # ── Print sweep table ─────────────────────────────────────────────────────
    print('STEP 5 — Panel Tilt Sensitivity Analysis (0°–60°)')
    print('=' * 58)
    print(f'  {"Tilt (°)":>8}  {"Annual Energy (kWh)":>20}  {"vs Flat (%)":>12}  {" ":>4}')
    print('  ' + '-' * 50)
    for _, row in sensitivity_df.iterrows():
        tilt    = int(row['tilt_deg'])
        energy  = row['annual_energy_kwh']
        pct     = row['pct_vs_flat']
        marker  = '  ◄ OPTIMAL' if tilt == optimal_tilt else ''

        if ALREADY_INSTALLED and tilt == current_tilt_analyzed: # Mark the closest analyzed tilt
            marker = f'  ◄ YOUR INPUT (~{int(PANEL_TILT_ANGLE_DEG)}°)'

        bar_len = int((energy / optimal_energy) * 20)
        print(f'  {tilt:>8}°  {energy:>20,.1f}  {pct:>+11.2f}%  '
              f'{"▓" * bar_len}{marker}')

    print('=' * 58)
    print(f'  Optimal tilt angle  : {optimal_tilt}°')
    print(f'  Max annual energy   : {optimal_energy:,.1f} kWh/year')
    print()

    # ── Installed user: current vs optimal comparison ─────────────────────────
    if ALREADY_INSTALLED:
        # Recalculate coverage based on current_energy_at_closest_tilt for consistency
        curr_cov  = round(min(current_energy_at_closest_tilt / cons * 100, 100), 1)

        gain_kwh  = optimal_energy - current_energy_at_closest_tilt
        gain_pct  = (gain_kwh / current_energy_at_closest_tilt) * 100
        new_cov   = round(min(optimal_energy / cons * 100, 100), 1)

        print(f'  ── Current vs Optimal ────────────────────────────────')
        print(f'  Your input tilt ({int(PANEL_TILT_ANGLE_DEG):>2}°) analyzed at {current_tilt_analyzed}° : {current_energy_at_closest_tilt:>10,.1f} kWh/year  '
              f'({curr_cov}% coverage)')
        print(f'  Optimal tilt  ({optimal_tilt:>2}°) : {optimal_energy:>10,.1f} kWh/year  '
              f'({new_cov}% coverage)')
        print(f'  Energy gain          : +{gain_kwh:>9,.1f} kWh/year  (+{gain_pct:.1f}%)')
        print()
        if gain_pct > 1.0: # Changed from > 2 to > 1.0
            print(f'  🔧 ACTION: Adjusting tilt from {int(PANEL_TILT_ANGLE_DEG)}° → {optimal_tilt}° '
                  f'gains {int(gain_kwh):,d} kWh/year')
            print(f'             without buying any new hardware.')
        else:
            print(f'  ✅ Your current tilt ({int(PANEL_TILT_ANGLE_DEG)}°) is already near-optimal, '
                  f'or very close to it. No major adjustment needed.')

    # ── Planning user: recommend best install tilt ────────────────────────────
    else:
        print(f'  💡 RECOMMENDATION: Install your panels at {optimal_tilt}° tilt')
        print(f'     for maximum annual energy yield of {optimal_energy:,.1f} kWh/year.')

    print('=' * 58)

### Section 19 · Step 6 — Scenario-Based Forecasting (Best / Worst / Average)

Environmental uncertainty is quantified from the spread across all 200 Random Forest trees:
- **Best case (90th percentile)** — high irradiance, clear sky conditions
- **Average (mean)** — model's central prediction
- **Worst case (10th percentile)** — low irradiance, cloudy / adverse conditions

Tilt correction is applied for already-installed users so scenarios reflect their actual setup.

In [ ]:
if not PLANNING_TO_INSTALL:
    print('ℹ️  Skipping — no analysis requested.')
else:

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 6 — SCENARIO-BASED FORECASTING (BEST / WORST / AVERAGE)
    #
    # Accounts for environmental uncertainty by drawing on the spread across
    # all 200 Random Forest trees:
    #   • Average    → mean prediction across all trees
    #   • Worst case → 10th percentile (low irradiance / cloudy conditions)
    #   • Best case  → 90th percentile (high irradiance / clear sky conditions)
    #
    # For 'installed' users → tilt correction is applied before computing
    #                         scenarios so results reflect their actual setup
    # For 'plan' users      → flat surface (0°) used as tilt not yet chosen
    # ══════════════════════════════════════════════════════════════════════════

    ghi_mean_vals  = future_df['GHI_mean'].values
    ghi_lower_vals = future_df['GHI_lower'].values
    ghi_upper_vals = future_df['GHI_upper'].values
    sza_vals       = future_df['Solar Zenith Angle'].values

    # ── Apply tilt correction if panels already installed ─────────────────────
    if ALREADY_INSTALLED:
        tilt_used      = PANEL_TILT_ANGLE_DEG
        ghi_mean_vals  = apply_tilt_correction(ghi_mean_vals,  sza_vals, tilt_used)
        ghi_lower_vals = apply_tilt_correction(ghi_lower_vals, sza_vals, tilt_used)
        ghi_upper_vals = apply_tilt_correction(ghi_upper_vals, sza_vals, tilt_used)
        tilt_label     = f'tilt-corrected at {tilt_used}°'
    else:
        tilt_used  = 0
        tilt_label = 'flat surface (0° — tilt not yet chosen)'

    # ── Compute per-year scenario table ───────────────────────────────────────
    scenario_rows = []
    for yr in FORECAST_YEARS:
        mask  = (future_df['Year'] == yr).values

        e_avg   = ghi_to_kwh(ghi_mean_vals[mask].sum())
        e_worst = ghi_to_kwh(ghi_lower_vals[mask].sum())
        e_best  = ghi_to_kwh(ghi_upper_vals[mask].sum())

        scenario_rows.append({
            'Year'         : yr,
            'Average_kWh'  : e_avg,
            'Worst_kWh'    : e_worst,
            'Best_kWh'     : e_best,
            'Avg_Cov_%'    : round(min(e_avg   / cons * 100, 100), 1),
            'Worst_Cov_%'  : round(min(e_worst / cons * 100, 100), 1),
            'Best_Cov_%'   : round(min(e_best  / cons * 100, 100), 1),
            'Surplus_kWh'  : round(e_avg - cons, 1),
        })

    scenario_df = pd.DataFrame(scenario_rows)

    # 5-year averages
    mean_avg   = scenario_df['Average_kWh'].mean()
    mean_worst = scenario_df['Worst_kWh'].mean()
    mean_best  = scenario_df['Best_kWh'].mean()

    cov_avg    = round(min(mean_avg   / cons * 100, 100), 1)
    cov_worst  = round(min(mean_worst / cons * 100, 100), 1)
    cov_best   = round(min(mean_best  / cons * 100, 100), 1)

    # ── Print per-year scenario table ─────────────────────────────────────────
    print('STEP 6 — Scenario-Based Forecasting')
    print(f'         ({tilt_label})')
    print('=' * 72)
    print(f'  {"Year":>4}  {"Average":>10}  {"Worst":>10}  {"Best":>10}  '
          f'{"Avg Cov":>8}  {"Worst Cov":>10}  {"Best Cov":>9}')
    print(f'  {"":>4}  {" (kWh)":>10}  {" (kWh)":>10}  {" (kWh)":>10}  '
          f'{" (%)":>8}  {" (%)":>10}  {" (%)":>9}')
    print('  ' + '-' * 66)
    for _, row in scenario_df.iterrows():
        print(f'  {int(row["Year"]):>4}  '
              f'{row["Average_kWh"]:>10,.1f}  '
              f'{row["Worst_kWh"]:>10,.1f}  '
              f'{row["Best_kWh"]:>10,.1f}  '
              f'{row["Avg_Cov_%"]:>7.1f}%  '
              f'{row["Worst_Cov_%"]:>9.1f}%  '
              f'{row["Best_Cov_%"]:>8.1f}%')
    print('  ' + '-' * 66)
    print(f'  {"5-Yr":>4}  '
          f'{mean_avg:>10,.1f}  '
          f'{mean_worst:>10,.1f}  '
          f'{mean_best:>10,.1f}  '
          f'{cov_avg:>7.1f}%  '
          f'{cov_worst:>9.1f}%  '
          f'{cov_best:>8.1f}%')
    print('=' * 72)

    # ── 5-year average summary ─────────────────────────────────────────────────
    print()
    print('  5-Year Average Scenario Summary:')
    print(f'  🌤  Average case  : {mean_avg:>9,.1f} kWh/year  →  {cov_avg}% of consumption covered')
    print(f'  🌧  Worst case    : {mean_worst:>9,.1f} kWh/year  →  {cov_worst}% of consumption covered')
    print(f'  ☀️   Best case     : {mean_best:>9,.1f} kWh/year  →  {cov_best}% of consumption covered')
    print()

    surplus_avg = mean_avg - cons
    if surplus_avg >= 0:
        print(f'  ✅ Even in average conditions your system generates a surplus')
        print(f'     of {int(surplus_avg):,d} kWh/year — excess can be fed back to the grid.')
    else:
        print(f'  ⚠️  Average case still needs {int(abs(surplus_avg)):,d} kWh/year from the grid.')
        print(f'     Best case covers {cov_best}% — good conditions nearly close the gap.')

    print()
    print(f'  Uncertainty spread  : ±{int((mean_best - mean_worst)/2):,d} kWh/year around the average')
    print(f'  Range               : {mean_worst:,.1f} – {mean_best:,.1f} kWh/year')
    print('=' * 72)

### Section 20 · Step 7 — Prediction Reliability Analysis (Hour vs Day Ahead)

Compares prediction accuracy across two time horizons:
- **Hour-ahead** — model predicts on test features as-is (most accurate)
- **Day-ahead** — simulated by adding ±5% Gaussian noise to weather inputs

This shows users that **short-term predictions are more reliable** than day-ahead forecasts,  
building trust by clearly stating model limitations.

In [ ]:
if not PLANNING_TO_INSTALL:
    print('ℹ️  Skipping — no analysis requested.')
else:

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 7 — PREDICTION RELIABILITY ANALYSIS
    #
    # Hour-ahead : predict on test set features as-is (baseline accuracy)
    # Day-ahead  : inject ±5% Gaussian noise to weather features to simulate
    #              the uncertainty in 24h-ahead weather forecasts
    # ══════════════════════════════════════════════════════════════════════════

    # ── Hour-ahead prediction ─────────────────────────────────────────────────
    y_pred_1h = global_model.predict(X_test)
    rmse_1h   = float(np.sqrt(mean_squared_error(y_test, y_pred_1h)))
    mae_1h    = float(mean_absolute_error(y_test, y_pred_1h))
    r2_1h     = float(r2_score(y_test, y_pred_1h))

    # ── Day-ahead prediction (weather noise simulation) ───────────────────────
    weather_cols = ['Temperature', 'Clearsky GHI', 'Relative Humidity', 'Wind Speed']
    X_day  = X_test.copy()
    noise  = np.random.normal(loc=1.0, scale=0.05, size=(len(X_day), len(weather_cols)))
    X_day[weather_cols] = np.clip(X_day[weather_cols].values * noise, 0, None)

    y_pred_24h = global_model.predict(X_day)
    rmse_24h   = float(np.sqrt(mean_squared_error(y_test, y_pred_24h)))
    mae_24h    = float(mean_absolute_error(y_test, y_pred_24h))
    r2_24h     = float(r2_score(y_test, y_pred_24h))

    degradation_rmse = (rmse_24h - rmse_1h) / rmse_1h * 100
    degradation_mae  = (mae_24h  - mae_1h)  / mae_1h  * 100

    print('STEP 7 — Prediction Reliability Analysis')
    print('=' * 58)
    print(f'  {"Metric":<10}  {"Hour-ahead":>12}  {"Day-ahead":>12}  {"Degradation":>13}')
    print(f'  {"-"*10}  {"-"*12}  {"-"*12}  {"-"*13}')
    print(f'  {"RMSE":<10}  {rmse_1h:>10.4f}  {rmse_24h:>10.4f}  {degradation_rmse:>+11.2f}%')
    print(f'  {"MAE":<10}  {mae_1h:>10.4f}  {mae_24h:>10.4f}  {degradation_mae:>+11.2f}%')
    print(f'  {"R²":<10}  {r2_1h:>10.4f}  {r2_24h:>10.4f}')
    print()
    print(f'  📊 Insight: Hour-ahead RMSE = {rmse_1h:.2f} W/m²')
    print(f'              Day-ahead  RMSE = {rmse_24h:.2f} W/m²')
    print(f'              Accuracy drops {degradation_rmse:.1f}% for day-ahead forecasts.')
    print(f'              → Trust next-hour outputs more than next-day outputs.')
    print('=' * 58)

### Section 21 · Step 8 — Final Output Report

In [ ]:
if not PLANNING_TO_INSTALL:
    print('ℹ️  Skipping — no analysis requested.')
else:
    import textwrap

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 8 — FINAL OUTPUT REPORT
    # ══════════════════════════════════════════════════════════════════════════

    surplus_avg = mean_avg - USER_YEARLY_CONSUMPTION_KWH
    cov_avg     = round(min(mean_avg   / USER_YEARLY_CONSUMPTION_KWH * 100, 100), 1)
    cov_worst   = round(min(mean_worst / USER_YEARLY_CONSUMPTION_KWH * 100, 100), 1)
    cov_best    = round(min(mean_best  / USER_YEARLY_CONSUMPTION_KWH * 100, 100), 1)

    if PLANNING_TO_INSTALL:
        if surplus_avg >= 0:
            recommendation = (
                f'✅ HIGHLY FEASIBLE — Solar fully covers your consumption '
                f'with a surplus of {int(surplus_avg):,d} kWh/year. '
                f'Even worst-case covers {cov_worst:.1f}%.'
            )
        elif cov_avg >= 70:
            recommendation = (
                f'🟡 FEASIBLE — Covers ~{cov_avg:.1f}% on average. '
                f'You still need {int(abs(surplus_avg)):,d} kWh/year from the grid. '
                f'Consider increasing panel area for full coverage.'
            )
        else:
            recommendation = (
                f'🔴 PARTIALLY FEASIBLE — Covers only ~{cov_avg:.1f}%. '
                f'With {USER_AREA_M2} m² you reduce grid dependency '
                f'but cannot fully replace it. Consider a larger installation.'
            )
    else:  # ALREADY_INSTALLED
        nearest_row    = sensitivity_df.iloc[
            (sensitivity_df['tilt_deg'] - PANEL_TILT_ANGLE_DEG).abs().argsort().iloc[0]
        ]
        current_energy = float(nearest_row['annual_energy_kwh'])
        gain_kwh_opt   = optimal_energy - current_energy
        gain_pct_opt   = gain_kwh_opt / current_energy * 100
        new_cov        = round(min(optimal_energy / USER_YEARLY_CONSUMPTION_KWH * 100, 100), 1)
        if gain_pct_opt > 1.0: # Changed from > 2 to > 1.0
            recommendation = (
                f'🔧 OPTIMISE YOUR TILT — Adjusting from {PANEL_TILT_ANGLE_DEG}° → {optimal_tilt}° '
                f'gains {int(gain_kwh_opt):,d} kWh/year (+{gain_pct_opt:.1f}%), '
                f'raising coverage from {cov_avg:.1f}% → {new_cov:.1f}%. '
                f'No new hardware needed.'
            )
        else:
            recommendation = (
                f'✅ YOUR TILT IS NEAR-OPTIMAL — Current {PANEL_TILT_ANGLE_DEG}° is close to '
                f'the optimal {optimal_tilt}°. System covering {cov_avg:.1f}% of your consumption.'
            )

    print('╔' + '═'*64 + '╗')
    print('║' + '  ☀️  SOLAR DECISION SUPPORT — FINAL REPORT'.center(64) + '║')
    print('╠' + '═'*64 + '╣')
    print(f'║  Location           : {UNSEEN_LOC_NAME:<41}║')
    print(f'║  Analysis type      : {"Planning to install" if PLANNING_TO_INSTALL else "Already installed":<41}║')
    print(f'║  Panel area         : {str(USER_AREA_M2) + " m²":<41}║')
    print(f'║  Yearly consumption : {str(f"{int(USER_YEARLY_CONSUMPTION_KWH):,d} kWh/year"):<41}║')
    if ALREADY_INSTALLED:
        print(f'║  Current tilt       : {str(f"{PANEL_TILT_ANGLE_DEG}°"):<41}║')
        print(f'║  System capacity    : {str(f"{INSTALLED_SYSTEM_CAPACITY_KW} kW"):<41}║')
    print('╠' + '═'*64 + '╣')
    print('║  PREDICTED GENERATION (avg 2025–2029)                         ║')
    print(f'║    Next-hour output   : {nh_kwh_avg:.5f} kWh{" "*34}║')
    print(f'║    Next-day  output   : {nd_kwh_avg:.3f} kWh{" "*34}║')
    print(f'║    Yearly   average   : {mean_avg:>10,.1f} kWh/year{" "*24}║')
    print(f'║    Yearly   worst     : {mean_worst:>10,.1f} kWh/year{" "*24}║')
    print(f'║    Yearly   best      : {mean_best:>10,.1f} kWh/year{" "*24}║')
    print('╠' + '═'*64 + '╣')
    print('║  COVERAGE & SCENARIOS                                         ║')
    print(f'║    🌤  Average case   : {cov_avg:>5.1f}% of consumption covered{" "*18}║')
    print(f'║    🌧  Worst case     : {cov_worst:>5.1f}% of consumption covered{" "*18}║')
    print(f'║    ☀️   Best case      : {cov_best:>5.1f}% of consumption covered{" "*18}║')
    surplus_label = 'Surplus' if surplus_avg >= 0 else 'Deficit'
    print(f'║    ⚡  {surplus_label:<11}    : {int(abs(surplus_avg)):>8,d} kWh/year{" "*22}║')
    print('╠' + '═'*64 + '╣')
    print('║  TILT OPTIMISATION                                            ║')
    print(f'║    Optimal tilt angle  : {optimal_tilt}°{" "*38}║')
    print(f'║    Energy at optimal   : {optimal_energy:>10,.1f} kWh/year{" "*24}║')
    print('╠' + '═'*64 + '╣')
    print('║  PREDICTION RELIABILITY                                       ║')
    print(f'║    Hour-ahead RMSE   : {rmse_1h:>8.2f} W/m²{" "*30}║')
    print(f'║    Day-ahead  RMSE   : {rmse_24h:>8.2f} W/m²  ({degradation_rmse:+.1f}%){' '*16}║')
    print('╠' + '═'*64 + '╣')
    print('║  RECOMMENDATION                                               ║')
    for line in textwrap.wrap(recommendation, 60):
        print(f'║  {line:<62}║')
    print('╚' + '═'*64 + '╝')

### Section 22 · Step 8 — Final Dashboard Visualisation

In [ ]:
if not PLANNING_TO_INSTALL:
    print('ℹ️  Skipping — no analysis requested.')
else:
    x_yrs = np.arange(len(FORECAST_YEARS))

    fig = plt.figure(figsize=(18, 14))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ── [0,0] Next-hour vs Next-day output ────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    horizons = ['Hour\nAvg','Hour\nWorst','Hour\nBest',
                'Day\nAvg', 'Day\nWorst', 'Day\nBest']
    values   = [nh_kwh_avg, nh_kwh_worst, nh_kwh_best,
                nd_kwh_avg, nd_kwh_worst, nd_kwh_best]
    colors_h = ['#2980b9','#e74c3c','#27ae60','#2980b9','#e74c3c','#27ae60']
    b1 = ax1.bar(range(6), values, color=colors_h, alpha=0.85)
    ax1.set_xticks(range(6)); ax1.set_xticklabels(horizons, fontsize=8)
    ax1.set_ylabel('kWh')
    ax1.set_title('Step 3: Hour & Day\nPower Output', fontweight='bold', fontsize=10)
    for b, v in zip(b1, values):
        ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.0001,
                 f'{v:.4f}', ha='center', fontsize=7, rotation=45)
    ax1.grid(True, axis='y', alpha=0.3)

    # ── [0,1] Yearly scenario grouped bars ───────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    w = 0.25
    ax2.bar(x_yrs-w, scenario_df['Worst_kWh'],   w, color='#e74c3c', alpha=0.85, label='Worst')
    ax2.bar(x_yrs,   scenario_df['Average_kWh'],  w, color='#2980b9', alpha=0.85, label='Average')
    ax2.bar(x_yrs+w, scenario_df['Best_kWh'],     w, color='#27ae60', alpha=0.85, label='Best')
    ax2.axhline(USER_YEARLY_CONSUMPTION_KWH, color='black', lw=1.8, ls='--', label='Consumption')
    ax2.set_xticks(x_yrs); ax2.set_xticklabels([str(y) for y in FORECAST_YEARS], fontsize=9)
    ax2.set_ylabel('kWh/year')
    ax2.set_title('Step 6: Annual Scenarios', fontweight='bold', fontsize=10)
    ax2.legend(fontsize=8); ax2.grid(True, axis='y', alpha=0.3)

    # ── [0,2] Coverage donut ──────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    cov_plot = min(cov_avg, 100)
    ax3.pie([cov_plot, max(0,100-cov_plot)],
            labels=['Solar\nCoverage','Grid'],
            colors=['#f39c12','#bdc3c7'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'width':0.55}, textprops={'fontsize':10})
    ax3.set_title('Step 4: Energy Coverage\n(Average Case)', fontweight='bold', fontsize=10)

    # ── [1,0] Tilt sensitivity curve ─────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.plot(sensitivity_df['tilt_deg'], sensitivity_df['annual_energy_kwh'],
             'o-', color='#8e44ad', lw=2, ms=6)
    ax4.axvline(optimal_tilt, color='#27ae60', lw=2, ls='--', label=f'Optimal {optimal_tilt}°')
    if ALREADY_INSTALLED:
        ax4.axvline(PANEL_TILT_ANGLE_DEG, color='#e74c3c', lw=2, ls='--',
                    label=f'Current {PANEL_TILT_ANGLE_DEG}°')
    ax4.set_xlabel('Tilt Angle (°)'); ax4.set_ylabel('Annual Energy (kWh)')
    ax4.set_title('Step 5: Tilt Sensitivity\n(0°–60°)', fontweight='bold', fontsize=10)
    ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

    # ── [1,1] Coverage % trend ────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.fill_between(x_yrs, scenario_df['Worst_Cov_%'], scenario_df['Best_Cov_%'],
                     alpha=0.2, color='#27ae60', label='Worst–Best band')
    ax5.plot(x_yrs, scenario_df['Avg_Cov_%'], 'o-', color='#27ae60', lw=2, ms=7, label='Average')
    ax5.axhline(100, color='black', lw=1.2, ls='--', alpha=0.5)
    ax5.set_xticks(x_yrs); ax5.set_xticklabels([str(y) for y in FORECAST_YEARS], fontsize=9)
    ax5.set_ylim(0, 115); ax5.set_ylabel('Coverage (%)')
    ax5.set_title('Step 6: % Consumption\nCovered', fontweight='bold', fontsize=10)
    for i, row in scenario_df.iterrows():
        ax5.annotate(f"{row['Avg_Cov_%']:.1f}%", (i, row['Avg_Cov_%']+2),
                     ha='center', fontsize=9, fontweight='bold')
    ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

    # ── [1,2] Surplus / Deficit ───────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[1, 2])
    surp  = scenario_df['Average_kWh'] - USER_YEARLY_CONSUMPTION_KWH
    col6  = ['#27ae60' if v>=0 else '#e74c3c' for v in surp]
    ax6.bar(x_yrs, surp, color=col6, alpha=0.85)
    ax6.axhline(0, color='black', lw=1.2)
    ax6.set_xticks(x_yrs); ax6.set_xticklabels([str(y) for y in FORECAST_YEARS], fontsize=9)
    ax6.set_ylabel('kWh/year')
    ax6.set_title('Step 6: Annual Surplus (+)\n/ Deficit (−)', fontweight='bold', fontsize=10)
    for i, v in enumerate(surp):
        # Corrected format specifier: convert to int before applying comma formatting
        ax6.text(i, v+(20 if v>=0 else -80), f'{int(v):,d}', ha='center', fontsize=9)
    ax6.grid(True, axis='y', alpha=0.3)

    # ── [2,0:2] Monthly GHI time-series ──────────────────────────────────────
    ax7 = fig.add_subplot(gs[2, 0:2])
    monthly = future_df.groupby(['Year','Month'])[['GHI_mean','GHI_lower','GHI_upper']].mean().reset_index()
    monthly['date'] = pd.to_datetime(monthly[['Year','Month']].assign(Day=15))
    ax7.fill_between(monthly['date'], monthly['GHI_lower'], monthly['GHI_upper'],
                     alpha=0.25, color='#e67e22', label='Worst–Best range')
    ax7.plot(monthly['date'], monthly['GHI_mean'], color='#e67e22', lw=2, label='Average GHI')
    for yr in FORECAST_YEARS[1:]:
        ax7.axvline(pd.Timestamp(f'{yr}-01-01'), color='grey', lw=0.7, ls='--', alpha=0.5)
    ax7.set_ylabel('Monthly Avg GHI (W/m²)')
    ax7.set_title('5-Year GHI Forecast (2025–2029) — Monthly Average with Scenario Band',
                  fontweight='bold', fontsize=10)
    ax7.legend(fontsize=9); ax7.grid(True, alpha=0.3)

    # ── [2,2] Reliability RMSE bar ────────────────────────────────────────────
    ax8 = fig.add_subplot(gs[2, 2])
    b8 = ax8.bar(['Hour-ahead','Day-ahead'], [rmse_1h, rmse_24h],
                 color=['#27ae60','#e74c3c'], alpha=0.85, width=0.5)
    for bar, val in zip(b8, [rmse_1h, rmse_24h]):
        ax8.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
    ax8.set_ylabel('RMSE (W/m²)')
    ax8.set_title(f'Step 7: Reliability\n(+{degradation_rmse:.1f}% RMSE day-ahead)',
                  fontweight='bold', fontsize=10)
    ax8.grid(True, axis='y', alpha=0.3)

    fig.suptitle(
        f'☀️  Solar Decision Support Dashboard — {UNSEEN_LOC_NAME}  '
        f'({"Planning to Install" if PLANNING_TO_INSTALL else "Already Installed"})',
        fontsize=14, fontweight='bold', y=1.01
    )
    plt.savefig(OUTPUT_DIR / 'final_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Dashboard saved → outputs/final_dashboard.png')

### Section 23 · Save All Outputs

In [ ]:
# Save forecast predictions
future_df.to_csv(OUTPUT_DIR / 'forecast_2025_2029.csv', index=False)
print(f'✅ Forecast saved          → {OUTPUT_DIR}/forecast_2025_2029.csv')

# Save tilt sensitivity table
if 'sensitivity_df' in dir():
    sensitivity_df.to_csv(OUTPUT_DIR / 'tilt_sensitivity.csv', index=False)
    print(f'✅ Tilt sensitivity saved  → {OUTPUT_DIR}/tilt_sensitivity.csv')

# Save scenario forecast table
if 'scenario_df' in dir():
    scenario_df.to_csv(OUTPUT_DIR / 'scenario_forecast.csv', index=False)
    print(f'✅ Scenario forecast saved → {OUTPUT_DIR}/scenario_forecast.csv')

print(f'\n📁 All outputs in: {OUTPUT_DIR.resolve()}')
print('   global_rf_model.pkl            — trained global model')
print('   forecast_2025_2029.csv         — hourly GHI predictions (mean/worst/best)')
print('   tilt_sensitivity.csv           — tilt sweep 0°–60° results')
print('   scenario_forecast.csv          — per-year scenario table')
print('   global_model_pred_vs_actual.png— model accuracy scatter chart')
print('   feature_importance.png         — feature importance chart')
print('   forecast_5year.png             — 5-year GHI forecast chart')
print('   final_dashboard.png            — 9-panel decision support dashboard')
